# Introduction to Semantic Segmentation
## CSCI E-25
### Steve Elston    

## Introduction

**Semantic segmentation** is one of three types of image segmentation tasks. The goal of semantic segmentation is to label each of the pixels in an image by category of the 'thing' associated with each pixel. In other words, the labeling associates the semantics of the things in the image with pixel labels.     

You will use a convolution neural network with an encoder-decoder atchitecture known as [**DeepLab3+**](https://arxiv.org/abs/1802.02611v3) to perform multi-class semantic segmentation. For this exercise you will use the [**Crowd Instance-level Human Parsing CHIP Dataset**](https://github.com/nikhilroxtomar/Multiclass-Segmentation-on-Crowd-Instance-level-Human-Parsing-CHIP-Dataset-using-UNET) data set to segment images with multiple people in to up to 20 semmantic categories. The categories include face, hair, arms and hands, legs, feet, etc. The dataset contains pairs of an $512 \times 512 \times 3$ RGB image and an $512 \times 512 \times 1$ segmentation mask. The mask labels pixels into one of 20 categories. Class 0 is the background task. A description of the other 19 class labels can be found [here](https://datasetninja.com/cihp).   

### References

The example in this notebook is based on the work in the these classic research papers.  
- [Encoder-Decoder with Atrous Separable Convolution for Semantic Image Segmentation, Chen, et. al. 2018](https://arxiv.org/abs/1802.02611)
- [Rethinking Atrous Convolution for Semantic Image Segmentation, Chen et. al., 2017](https://arxiv.org/abs/1706.05587)
- [DeepLab: Semantic Image Segmentation with Deep Convolutional Nets, Atrous Convolution, and Fully Connected CRFs, Chen et. al., 2018](https://arxiv.org/abs/1606.00915)

### Attributions    
This code in this notebook is based on on a [Keras example notebook](https://keras.io/examples/vision/deeplabv3_plus/) by [Soumik Rakshit](http://github.com/soumik12345), and posted 2024/01/05.   

Portions of code in this notebook were generated with Microsoft Copilot and Google Gemini.    

## Setup    

It is recommended that you run this notebook in [**Google Colab**](https://colab.research.google.com/signup). A payed subscription will generally provide faster execution. This notebook is intended to be run with GPU acceleration and the HighRAM setting. Model training in this notebook is expected to take about one hour. Download of the dataset can take some time.      

To get started, execute the code in the cell below to import the required packages.

In [ ]:
import keras
from keras import layers
from keras import ops
from keras.optimizers.schedules import ExponentialDecay

import os
import numpy as np
import pandas as pd
from glob import glob
import cv2
from scipy.io import loadmat
import matplotlib.pyplot as plt

# For data preprocessing
import tensorflow as tf
from tensorflow import image as tf_image
from tensorflow import data as tf_data
from tensorflow import io as tf_io

### Downloading the data

We will use the [Crowd Instance-level Human Parsing Dataset](https://arxiv.org/abs/1811.12596)
for training our model. The Crowd Instance-level Human Parsing (CIHP) dataset has 38,280 diverse human images. Each image in CIHP is labeled with pixel-wise annotations for 20 categories, as well as instance-level identification.
This dataset can be used for the "human part segmentation" task.

The code in the cell below downloads dataset and unzips the files. The download and unzip operations only occur if the dataset is not found at the path specified. Execute this code.    

> *Note:* The code below includes a path to the zip file in a GoogleDrive location. The commented out path is to the original dataset archive. 

In [ ]:
zip_file_path = "instance-level-human-parsing.zip"
unzipped_dir_path = "instance-level_human_parsing/instance-level_human_parsing"

if not os.path.exists(zip_file_path) and not os.path.exists(unzipped_dir_path):
    print("Downloading dataset...")
#    !gdown https://drive.google.com/uc?id=1B9A9UCJYMwTL4oBEo4RZfbMZMaZhKJaz # Path to original dataset archive 
    !gdown https://drive.google.com/uc?id=1B9A9UCJYMwTL4oBEo4RZfbMZMaZhKJaz # Path to file in instructor GoogleDrive
    print("Unzipping files.")
    !unzip -q instance-level-human-parsing.zip
    print("Dataset downloaded and unzipped.")
elif os.path.exists(zip_file_path) and not os.path.exists(unzipped_dir_path):
    print("Zip file found, unzipping dataset...")
    !unzip -q instance-level-human-parsing.zip
    print("Dataset unzipped.")
else:
    print("Dataset already available. Skipping download and unzip.")

### Sampling and creating a TensorFlow dataset

The entire dataset contains 38,280 images. Training with this large a dataset requires significant computing resources. Since we are fine-tuning a pre-trained model we will use only 1000 images for training and 100 images for validation.

Execute the code in the cell below to sample the image dataset and to convert the sample to TensorFlow format.

In [ ]:
IMAGE_SIZE = 512
BATCH_SIZE = 4
NUM_CLASSES = 20
DATA_DIR = "./instance-level_human_parsing/instance-level_human_parsing/Training"
NUM_TRAIN_IMAGES = 1000
NUM_VAL_IMAGES = 100

train_images = sorted(glob(os.path.join(DATA_DIR, "Images/*")))[:NUM_TRAIN_IMAGES]
train_masks = sorted(glob(os.path.join(DATA_DIR, "Category_ids/*")))[:NUM_TRAIN_IMAGES]
val_images = sorted(glob(os.path.join(DATA_DIR, "Images/*")))[
    NUM_TRAIN_IMAGES : NUM_VAL_IMAGES + NUM_TRAIN_IMAGES
]
val_masks = sorted(glob(os.path.join(DATA_DIR, "Category_ids/*")))[
    NUM_TRAIN_IMAGES : NUM_VAL_IMAGES + NUM_TRAIN_IMAGES
]


def read_image(image_path, mask=False):
    image = tf_io.read_file(image_path)
    if mask:
        image = tf_image.decode_png(image, channels=1)
        image.set_shape([None, None, 1])
        image = tf_image.resize(images=image, size=[IMAGE_SIZE, IMAGE_SIZE])
    else:
        image = tf_image.decode_png(image, channels=3)
        image.set_shape([None, None, 3])
        image = tf_image.resize(images=image, size=[IMAGE_SIZE, IMAGE_SIZE])
    return image


def load_data(image_list, mask_list):
    image = read_image(image_list)
    mask = read_image(mask_list, mask=True)
    return image, mask


def data_generator(image_list, mask_list):
    dataset = tf_data.Dataset.from_tensor_slices((image_list, mask_list))
    dataset = dataset.map(load_data, num_parallel_calls=tf_data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)
    return dataset


train_dataset = data_generator(train_images, train_masks)
val_dataset = data_generator(val_images, val_masks)

print("Train Dataset:", train_dataset)
print("Val Dataset:", val_dataset)

### Exploring the dataset

To get the feel for the dataset the code in the cell below displays the first 10 images along with their segmentation masks. The dataset contains pairs of a $512 \times 512 \times 3$ RGB image and a $512 \times 512 \times 1$ ground truth segmentation mask. This segmentation mask contains the labels for training the multi-class segmentation model. The 20 categories of pixel labels are color coded for display.  

Execute this code and examine the results.

In [ ]:
for i in range(10):
  _, ax = plt.subplots(1,2, figsize=(8, 4))
  example_image, example_mask = load_data(train_images[i], train_masks[i])
  ax[0].imshow(example_image.numpy().astype(np.uint8))
  ax[1].imshow(example_mask)
  plt.show()

While examining the segmentation masks shown above notice the large class imbalance. This can be see as the ratio between the background label (a deep blue) and the other labels. One can also see relative differences in the labeled area of the people shown, such as the difference is size of feet vs. torso.

> **Exercise 9-1:** Examine the images and ground truth masks displayed above and answer these questions in one or a few sentences:       
> 1. The semantic segmentation ground truth mask labels are coded as colors in these plots. Compare the mask labels to the body parts of the people shown in the associated image. Do these ground truth labels have semantic meaning with relationship to the structure of these humans? Provide an example.
> 1. Are the semantic segmentation categories consistent from image to image and why is this important in training a model?   

> **Answers:**
> 1.               
> 2.               

## Building the DeepLabV3+ model

DeepLabv3+ uses the encoder-decoder architecture shown in the figure below.     

The encoder creates a multi-scale feature map using multiple dilated convolution layers to sample larger scales while maintaining spatial dimensionality. These layers use depth-wise convolution for efficiency. Combined with a pooling layer, this architectural component performs spatial pyramid pooling, [He, et. al., 2014](https://arxiv.org/abs/1406.4729).    

The decoder refines the segmentation results along object boundaries. The decoder is comprised of several transpose convolution layers to up-sample to higher spatial resolutions.

![](https://github.com/lattice-ai/DeepLabV3-Plus/raw/master/assets/deeplabv3_plus_diagram.png)
<div style="text-align: center;"> <b> Architecture of DeepLabV3+, Chen, et. al., 2018 </b> </div>

> **Note:** Before proceeding you might want to review the first two sections of [**Multi-Scale Context Aggregation by Dialated Convolutions**](https://arxiv.org/pdf/1511.07122v3) by Yu and  Koltun, 2016.  

> **Exercise 9-2:**
> 1. Why is semantic segmentation a dense task?
> 2. Why is in necessary to up-sample the pixel assignments to the same resolution as the original image?
> 3. Given the up-sampling, what can you say about possible accuracy problems along the class boundaries?
> 4. Explain how a generative model might be a helpful alternative when considering with the boundary problem?    

> **Answers:**
> 1.             
> 2.             
> 3.             
> 4.                   

### Building the spatial pyramid pooling class

The code in the cell below defines the `DilatedSpatialPyramidPooling` function. This function concatenates upsampled average pooled input with the four scales of down-sampled input.       

Execute this code to define this function.      

In [ ]:

def convolution_block(
    block_input,
    num_filters=256,
    kernel_size=3,
    dilation_rate=1,
    use_bias=False,
):
    x = layers.Conv2D(
        num_filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding="same",
        use_bias=use_bias,
        kernel_initializer=keras.initializers.HeNormal(),
    )(block_input)
    x = layers.BatchNormalization()(x)
    return ops.nn.relu(x)


def DilatedSpatialPyramidPooling(dspp_input):
    dims = dspp_input.shape
    x = layers.AveragePooling2D(pool_size=(dims[-3], dims[-2]))(dspp_input)
    x = convolution_block(x, kernel_size=1, use_bias=True)
    out_pool = layers.UpSampling2D(
        size=(dims[-3] // x.shape[1], dims[-2] // x.shape[2]),
        interpolation="bilinear",
    )(x)

    out_1 = convolution_block(dspp_input, kernel_size=1, dilation_rate=1)
    out_6 = convolution_block(dspp_input, kernel_size=3, dilation_rate=6)
    out_12 = convolution_block(dspp_input, kernel_size=3, dilation_rate=12)
    out_18 = convolution_block(dspp_input, kernel_size=3, dilation_rate=18)

    x = layers.Concatenate(axis=-1)([out_pool, out_1, out_6, out_12, out_18])
    output = convolution_block(x, kernel_size=1)
    return output


### Define the full model

The code in the cell below defines the `DeeplabV3Plus` model.
- The model uses a pretrained ResNet 50 as a backbone.     
- The backbone output is used as an input to the `DilatedSpatialPyramidPooling`.
- The spatially pooled tensor is bilinearly up-sampled by a factor 4.
- The up-sampled tensor, representing the large-scale features, is concatenated with the output from a selected layer in the backbone.
- Finally, several convolution layers are used and the result up-sampled to the spatial resolution of the input image.

Execute this code to instantiate this function and the model.  

In [ ]:
def DeeplabV3Plus(image_size, num_classes):
    model_input = keras.Input(shape=(image_size, image_size, 3))
    preprocessed = keras.applications.resnet50.preprocess_input(model_input)
    resnet50 = keras.applications.ResNet50(
        weights="imagenet", include_top=False, input_tensor=preprocessed
    )
    x = resnet50.get_layer("conv4_block6_2_relu").output
    x = DilatedSpatialPyramidPooling(x)

    input_a = layers.UpSampling2D(
        size=(image_size // 4 // x.shape[1], image_size // 4 // x.shape[2]),
        interpolation="bilinear",
    )(x)
    input_b = resnet50.get_layer("conv2_block3_2_relu").output
    input_b = convolution_block(input_b, num_filters=48, kernel_size=1)

    x = layers.Concatenate(axis=-1)([input_a, input_b])
    x = convolution_block(x)
    x = convolution_block(x)
    x = layers.UpSampling2D(
        size=(image_size // x.shape[1], image_size // x.shape[2]),
        interpolation="bilinear",
    )(x)
    model_output = layers.Conv2D(num_classes, kernel_size=(1, 1), padding="same")(x)
    return keras.Model(inputs=model_input, outputs=model_output)

model = DeeplabV3Plus(image_size=IMAGE_SIZE, num_classes=NUM_CLASSES)

## Training the Model

We are now ready to train the model. Since we are starting with a pretrained model only a few epochs are required to achieve a fully-trained model. The original code resulted in erratic training and validation loss and accuracy curves, indicating that the model was over-fit.Considerable experimentation was required to find a learning rate schedule and weight decay parameter value that gave a reasonable set of loss and accuracy curves. The updated code addresses this problem in two ways:     
     - The original learning rate appears to be too high. To address this problem a exponentially decaying learning rate schudule with a lower initial rate is used.       
     - The original optimizer specification did not include any weight decay. The updated code includes the `weight_decay` hyperparameter.    
Execute the code below and examine the results.   

In [ ]:
epochs = 15
initial_learning_rate=0.00001
decay_steps=500
decay_rate=0.9
weight_decay=0.2
learning_rate_schedule = ExponentialDecay(
    initial_learning_rate=initial_learning_rate,
    decay_steps=decay_steps,
    decay_rate=decay_rate
)
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=learning_rate_schedule,
                                     weight_decay=weight_decay),
                                     loss=loss,
                                     metrics=["accuracy"],
)

history = model.fit(train_dataset, validation_data=val_dataset, epochs=epochs)

def plot_loss(history):
    '''Function to plot the loss vs. epoch'''
    train_loss = history.history['loss']
    test_loss = history.history['val_loss']
    x = list(range(1, len(test_loss) + 1))
    plt.plot(x, test_loss, color = 'red', label = 'Test loss')
    plt.plot(x, train_loss, label = 'train losss')
    plt.legend()
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss vs. Epoch')
    plt.show()

def plot_accuracy(history):
    x = list(range(1, len(history.history['accuracy']) + 1))
    plt.plot(x, history.history['accuracy'], label = 'Training accuracy')
    plt.plot(x, history.history['val_accuracy'], color = 'red', label = 'Validation accuracy')
    plt.legend()
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Accuracy vs. Epoch')
    plt.show()

plot_loss(history)
plot_accuracy(history)

> **Exercise 9-3:** Answer the following questions in one or a few sentences.   
> 1. What is categorical cross entropy an appropriate loss function for this dense task.     
> 2. Examine the loss and accuracy curve. Explain the evidence you see that the fine-tune training of the model was compete and had not resulted in over-fitting    

> **Answer:**
> 1.               
> 2.               

## Model Evaluation          

With the model trained, we are ready to perform some detailed evaluation. The code in the cell below computes and displays performance metrics averaged across classes and for each class. Execute this code and examine the results.             

In [ ]:
def calculate_and_display_metrics(model, dataset, num_classes):
    # Accumulators for per-class metrics
    per_class_total_tp = np.zeros(num_classes)
    per_class_total_fp = np.zeros(num_classes)
    per_class_total_fn = np.zeros(num_classes)
    per_class_true_count = np.zeros(num_classes) # Accumulator for true pixel count for each class

    # Accumulators for overall pixel accuracy
    total_correct_pixels = 0
    total_pixels_processed = 0

    print("Calculating metrics...")

    # Iterate over the dataset
    for images, masks in dataset:
        # Perform inference to get raw logits
        predictions_logits = model.predict(images, verbose=0)
        # Convert logits to class predictions (0 to NUM_CLASSES-1)
        predicted_masks = np.argmax(predictions_logits, axis=-1)  # Shape: (BATCH_SIZE, IMAGE_SIZE, IMAGE_SIZE)

        # Process each image in the current batch
        for i in range(images.shape[0]):
            # Flatten ground truth mask and predicted mask for easier comparison
            y_true_single = masks[i].numpy().flatten()  # Shape: (IMAGE_SIZE*IMAGE_SIZE,)
            y_pred_single = predicted_masks[i].flatten()  # Shape: (IMAGE_SIZE*IMAGE_SIZE,)

            # Update overall pixel accuracy accumulators
            total_correct_pixels += np.sum(y_true_single == y_pred_single)
            total_pixels_processed += y_true_single.size

            # Calculate TP, FP, FN for each class
            for cls_id in range(num_classes):
                # True Positives: pixels correctly identified as class `cls_id`
                true_positives = np.sum((y_true_single == cls_id) & (y_pred_single == cls_id))
                # False Positives: pixels predicted as `cls_id` but are actually another class
                false_positives = np.sum((y_true_single != cls_id) & (y_pred_single == cls_id))
                # False Negatives: pixels that are `cls_id` but predicted as another class
                false_negatives = np.sum((y_true_single == cls_id) & (y_pred_single != cls_id))

                per_class_total_tp[cls_id] += true_positives
                per_class_total_fp[cls_id] += false_positives
                per_class_total_fn[cls_id] += false_negatives
                per_class_true_count[cls_id] += np.sum(y_true_single == cls_id) # Accumulate true pixel count for each class

    # Calculate final overall (micro-average) precision, recall, and Dice Loss
    final_overall_tp = np.sum(per_class_total_tp)
    final_overall_fp = np.sum(per_class_total_fp)
    final_overall_fn = np.sum(per_class_total_fn)

    final_overall_precision = final_overall_tp / (final_overall_tp + final_overall_fp) if (final_overall_tp + final_overall_fp) > 0 else 0
    final_overall_recall = final_overall_tp / (final_overall_tp + final_overall_fn) if (final_overall_tp + final_overall_fn) > 0 else 0

    overall_dice_denominator = (2 * final_overall_tp) + final_overall_fp + final_overall_fn
    final_overall_dice_loss = 1 - ((2 * final_overall_tp) / overall_dice_denominator) if overall_dice_denominator > 0 else np.nan

    # Calculate Overall Pixel Accuracy
    overall_pixel_accuracy = total_correct_pixels / total_pixels_processed if total_pixels_processed > 0 else 0

    print(f"\nOverall Pixel Accuracy: {overall_pixel_accuracy:.4f}") # Print this first as requested
    print(f"Overall (Micro-average) Precision: {final_overall_precision:.4f}")
    print(f"Overall (Micro-average) Recall: {final_overall_recall:.4f}")
    print(f"Overall (Micro-average) Dice Loss: {final_overall_dice_loss:.4f}")

    # Calculate final per-class precision, recall, Dice Loss, and IoU (accuracy)
    final_per_class_precision = np.zeros(num_classes)
    final_per_class_recall = np.zeros(num_classes)
    final_per_class_dice_loss = np.zeros(num_classes)
    final_per_class_iou = np.zeros(num_classes) # New: for class-specific accuracy

    for cls_id in range(num_classes):
        # Precision for current class
        denominator_precision = per_class_total_tp[cls_id] + per_class_total_fp[cls_id]
        if denominator_precision > 0:
            final_per_class_precision[cls_id] = per_class_total_tp[cls_id] / denominator_precision
        else:
            final_per_class_precision[cls_id] = np.nan # Undefined if no predictions for this class

        # Recall for current class
        denominator_recall = per_class_total_tp[cls_id] + per_class_total_fn[cls_id]
        if denominator_recall > 0:
            final_per_class_recall[cls_id] = per_class_total_tp[cls_id] / denominator_recall
        else:
            final_per_class_recall[cls_id] = np.nan # Undefined if no true instances of this class

        # Dice Coefficient and Dice Loss for current class
        denominator_dice = (2 * per_class_total_tp[cls_id]) + per_class_total_fp[cls_id] + per_class_total_fn[cls_id]
        if denominator_dice > 0:
            dice_coefficient = (2 * per_class_total_tp[cls_id]) / denominator_dice
            final_per_class_dice_loss[cls_id] = 1 - dice_coefficient
        else:
            final_per_class_dice_loss[cls_id] = np.nan # Undefined if no true or predicted pixels for this class

        # IoU (Jaccard Index) for current class - commonly used as class-specific accuracy
        iou_denominator = per_class_total_tp[cls_id] + per_class_total_fp[cls_id] + per_class_total_fn[cls_id]
        if iou_denominator > 0:
            final_per_class_iou[cls_id] = per_class_total_tp[cls_id] / iou_denominator
        else:
            final_per_class_iou[cls_id] = np.nan # Undefined if no true or predicted pixels for this class

    # Create a DataFrame for per-class metrics with desired column order
    class_ids = np.arange(num_classes)
    metrics_df = pd.DataFrame({
        'Class ID': class_ids,
        'True Pixel Count': per_class_true_count,
        'IoU (Accuracy)': final_per_class_iou, # New column, renamed for clarity as per user request
        'Precision': final_per_class_precision,
        'Recall': final_per_class_recall,
        'Dice Loss': final_per_class_dice_loss # New column
    })

    print("\nPer-class Metrics:")
    display(metrics_df.round(4))

    return metrics_df

metrics_df = calculate_and_display_metrics(model, val_dataset, NUM_CLASSES)

> **Exercise 9-4:** Examine these results and answer the following questions in one or a few sentences.
> 1. Examine the true pixel count column. Describe the severity of the class imbalance.
> 2. Explain how the one predominant class, the background, optimistically biases the overall or aggregated performance metrics.  
> 3. Notice that several of the classes have a Dice loss close to the maximum values, $DL \ge 0.95$. What can you say about the estimated masks for these classes, and what property do these classes have in common?

> **Answers:**
> 1.           
> 2.           
> 3.           

## Train Model With Focal Loss     

Given the foregoing results, the question is what can we do to improve them. One answer is to use **focal loss**, [Lin, et. al., 2017](https://arxiv.org/abs/1708.02002). The code in the cell below defines a focal loss function. Execute this code to instantiate this function.    

In [ ]:
@tf.keras.utils.register_keras_serializable()
class SparseFocalLoss(keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, from_logits=True, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha
        self.from_logits = from_logits

    def call(self, y_true, y_pred):
        # y_true: (batch,) or (batch, H, W) or (batch, H, W, 1) for segmentation
        # y_pred: logits of shape (..., num_classes)

        y_true = tf.cast(y_true, tf.int32)
        # Squeeze the last dimension of y_true, as it can be (BATCH_SIZE, H, W, 1)
        # We need (BATCH_SIZE, H, W) for indexing.
        # Only squeeze if y_true has 4 dimensions and the last is 1.
        if y_true.shape.ndims == 4 and y_true.shape[-1] == 1:
            squeezed_y_true = tf.squeeze(y_true, axis=-1) # Shape: (BATCH_SIZE, H, W)
        else:
            squeezed_y_true = y_true # Assume it's already (BATCH_SIZE, H, W) or (batch,)

        # Compute log-probabilities
        if self.from_logits:
            log_probs = tf.nn.log_softmax(y_pred, axis=-1) # Shape: (BATCH_SIZE, H, W, num_classes)
        else:
            log_probs = tf.math.log(tf.clip_by_value(y_pred, 1e-7, 1.0))

        # Gather log p_t for the true classes
        # params: log_probs (B, H, W, C)
        # indices: squeezed_y_true (B, H, W)
        # batch_dims: 3 (for B, H, W) to match the batch and spatial dimensions
        log_pt = tf.gather(log_probs, squeezed_y_true, batch_dims=squeezed_y_true.shape.ndims)
        # log_pt will have shape (B, H, W)

        pt = tf.exp(log_pt)

        # Focal loss
        focal_term = (1.0 - pt) ** self.gamma
        loss = -self.alpha * focal_term * log_pt

        return tf.reduce_mean(loss)

The code in the cell below does the following.     
- Instantiates a new model.      
- Trains the model using focal loss.     
- Computes and displays evaluation metrics.     

Execute this code and examine the results.    

In [ ]:
learning_rate_schedule = ExponentialDecay(
    initial_learning_rate=initial_learning_rate,
    decay_steps=decay_steps,
    decay_rate=decay_rate
)
loss = SparseFocalLoss(gamma=2.0,
                           alpha=0.25,
                           from_logits=True)

model2 = DeeplabV3Plus(image_size=IMAGE_SIZE, num_classes=NUM_CLASSES)
model2.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=learning_rate_schedule,
                                     weight_decay=weight_decay),
                                     loss=loss,
                                     metrics=["accuracy"],
)

history = model2.fit(train_dataset, validation_data=val_dataset, epochs=epochs)

plot_loss(history)
plot_accuracy(history)
metrics_df = calculate_and_display_metrics(model2, val_dataset, NUM_CLASSES)

> **Exercise 9-5:** Examine these results. The question one should consider is if using focal loss provided any improvement in model performance. Answer the following questions in one or a few sentences.
> 1. Has changing the loss function had any significant effect on the overall (aggregated performance metrics) compared to the model using cross entropy loss?
> 2. Focusing on the cases were dice loss is $\le 0.95$ describe any marginal improvement in performance that you see for the model trained with focal loss?
> 3. What can you say about the cases where training the model with focal loss has not improved high dice loss values.    

> **Answers:**
> 1.          
> 2.       
> 3.               

## Inference using Colormap Overlay

The raw predictions from the model represent a one-hot encoded tensor of shape `(N, 512, 512, 20)`where each one of the 20 channels is a binary mask corresponding to a predicted label. To visualize the results, the one-hot-encoded masks must be converted to RGB format with a unique color for each category. The colors corresponding to each label are found in the `human_colormap.mat` file provided as part of the dataset.    

The code in the cell below plots three images side by side for each sample, from left to right:    
- The original image.
- The image with the estimated segmentation mask overlaid.
- The segmentation mask.

### Inference on the training images      

Execute this code that samples the training images and examine the displayed results.   

In [ ]:
# Loading the Colormap
colormap = loadmat(
    "./instance-level_human_parsing/instance-level_human_parsing/human_colormap.mat"
)["colormap"]
colormap = colormap * 100
colormap = colormap.astype(np.uint8)


def infer(model, image_tensor):
    predictions = model.predict(np.expand_dims((image_tensor), axis=0))
    predictions = np.squeeze(predictions)
    predictions = np.argmax(predictions, axis=2)
    return predictions


def decode_segmentation_masks(mask, colormap, n_classes):
    r = np.zeros_like(mask).astype(np.uint8)
    g = np.zeros_like(mask).astype(np.uint8)
    b = np.zeros_like(mask).astype(np.uint8)
    for l in range(0, n_classes):
        idx = mask == l
        r[idx] = colormap[l, 0]
        g[idx] = colormap[l, 1]
        b[idx] = colormap[l, 2]
    rgb = np.stack([r, g, b], axis=2)
    return rgb


def get_overlay(image, colored_mask):
    image = keras.utils.array_to_img(image)
    image = np.array(image).astype(np.uint8)
    overlay = cv2.addWeighted(image, 0.35, colored_mask, 0.65, 0)
    return overlay


def plot_samples_matplotlib(display_list, figsize=(5, 3)):
    _, axes = plt.subplots(nrows=1, ncols=len(display_list), figsize=figsize)
    for i in range(len(display_list)):
        if display_list[i].shape[-1] == 3:
            axes[i].imshow(keras.utils.array_to_img(display_list[i]))
        else:
            axes[i].imshow(display_list[i])
    plt.show()


def plot_predictions(images_list, colormap, model):
    for image_file in images_list:
        image_tensor = read_image(image_file)
        prediction_mask = infer(image_tensor=image_tensor, model=model)
        prediction_colormap = decode_segmentation_masks(prediction_mask, colormap, 20)
        overlay = get_overlay(image_tensor, prediction_colormap)
        plot_samples_matplotlib(
            [image_tensor, overlay, prediction_colormap], figsize=(18, 14)
        )

plot_predictions(train_images[:10], colormap, model=model2)

### Visualizing errors in the learned masks     
One may well wonder about the error between the ground truth segmenation masks and the predicted segmentation masks. The code below does the following:   
1. From left to right the code displays the ground truth mask, the predicted mask and the absolute difference between the ground truth and predicted as images.
2. Compute and print an average error between the ground truth mask and the predicted mask. This average error is scaled to a range of $[0,1]$, with 0 being no error and 1 being maximum disagreement.

Execute the code in the cell below to examine the error map.   

In [ ]:
for i in range(5):
  val_image, val_mask = load_data(val_images[i], val_masks[i])
  predicted_mask = infer(model2, val_image)

  _ , ax = plt.subplots(1,3, figsize=(10, 3))
  ax[0].imshow(val_mask)
  ax[1].imshow(predicted_mask)

  predicted_mask_gray = cv2.cvtColor(
                                     decode_segmentation_masks(predicted_mask, colormap, 20),
                                      cv2.COLOR_BGR2GRAY
                                     )
  mask_diff = np.abs(np.subtract(val_mask.numpy().reshape((512,512)),
                           predicted_mask_gray.reshape((512,512))))
  ax[2].imshow(mask_diff)
  print('Average error = ', str(np.divide(np.sum(mask_diff), 512*512*255)))
  plt.show()

> **Exercise 9-6:** Examine the images displayed above. In each row left to right are the ground truth segmentation mask, the predicted mask, and the difference between the ground truth and the prediction. The brighter the color in the right hand image the greater the error. Notice also the average relative error on a $[0,1]$ scale is printed. What can you notice about the size of the high error regions compared to the size of the overall image and what does this tell you about possible problems with class imbalance.  

> **Answer:**               

### Inference on Validation Images     

Finally, you can perform inference on the validation images and view some sample results by executing the code in the cell below.   

In [ ]:
plot_predictions(val_images[:4], colormap, model=model2)

In [ ]:
# Optionally, to kill server and stop paying
# Uncomment and run the code below
#if 'google.colab' in sys.modules:
#    from google.colab import runtime
#    runtime.unassign()

##### Copyright, 2025, 2026, Stephen F Elston. All rights reserved.